# CodeBART: Neural Code Summarization with PLBART and CodeT5

**Task.** Given the body of a Python function, generate a one-sentence natural-language summary
(the first line of its docstring).

**Approach.** Two pre-trained code-aware encoder-decoder transformers, **PLBART** and **CodeT5**,
are fine-tuned on the Python subset of the **CodeXGLUE code-to-text** benchmark with a shared,
leakage-free training and evaluation pipeline, and compared with BLEU and ROUGE.

**Highlights**
- Leakage-safe data preparation: the model never sees the docstring it is asked to write, and the
  check is measured rather than assumed (Section 2.3).
- Honest model selection: the checkpoint with the best validation BLEU is kept, with early
  stopping, and the test set is scored exactly once per model.
- Fully reproducible: fixed seeds, pinned preprocessing, dynamic padding, every hyper-parameter in
  one place.
- Ready-to-use inference helper that applies the same preprocessing to arbitrary Python source.

### Contents
1. [Environment and configuration](#1.-Environment-and-configuration)
2. [Data](#2.-Data)
3. [Shared pipeline](#3.-Shared-pipeline)
4. [PLBART](#4.-PLBART)
5. [CodeT5](#5.-CodeT5)
6. [Results](#6.-Results)
7. [Inference on new code](#7.-Inference-on-new-code)
8. [Methodology notes](#8.-Methodology-notes)
9. [Conclusion and next steps](#9.-Conclusion-and-next-steps)


## 1. Environment and configuration

**Running on Kaggle (recommended).** This notebook was designed for a Kaggle GPU session:

- *Settings → Accelerator*: **GPU T4 x2** or **GPU P100**. Only one GPU is used.
- *Settings → Internet*: **On** (the dataset and model weights are downloaded from the Hugging Face Hub).
- Expected runtime on a T4 with the default 10% data subset: roughly 1.5-2.5 hours end to end
  (about 20 minutes with `SUBSET_FRACTION = 0.01`).
- Tested with Kaggle's current image (Python 3.13, PyTorch 2.11, transformers 5.20, datasets 4.8); the
  code also runs on transformers 4.4x.
- Fine-tuned checkpoints are written to `/kaggle/working/checkpoints` and appear in the notebook's
  *Output* tab, from where they can be downloaded or attached to another notebook as a dataset.

Everything specific to Kaggle is marked with a `# Kaggle:` comment in the code.

In [ ]:
# Kaggle: torch, transformers, datasets and nltk are pre-installed and already compatible with each
# other, so they are deliberately not upgraded here. Only the two missing packages are installed.
# The install needs "Internet: On" in the notebook settings.
%pip install -q rouge-score sentencepiece


In [ ]:
import ast
import io
import math
import tokenize as py_tokenize
from pathlib import Path
from statistics import mean

import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    PLBartForConditionalGeneration,
    PLBartTokenizer,
    get_linear_schedule_with_warmup,
    set_seed,
)
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu, sentence_bleu
from rouge_score import rouge_scorer, scoring

# ---- configuration -------------------------------------------------------------------------
SEED = 42
SUBSET_FRACTION = 0.1    # fraction of each split used. 0.1 -> ~25k training functions.
                         # Kaggle: ~1.5-2.5 h end to end on a T4 at 0.1; set 0.01 for a ~20 min smoke run.
                         # Runtime scales roughly linearly with this value.
MAX_SOURCE_LEN = 384     # tokens of code fed to the encoder
MAX_TARGET_LEN = 128     # tokens of summary produced by the decoder
NUM_EPOCHS = 10          # upper bound; early stopping normally ends training well before this
PATIENCE = 2             # epochs without validation-BLEU improvement before stopping
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
NUM_BEAMS = 4

# Kaggle: /kaggle/working is the only writable directory that is kept as notebook output.
OUTPUT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle/working").exists() else Path("checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import transformers, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | datasets {datasets.__version__}")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU detected")
print("checkpoints will be saved to:", OUTPUT_DIR.resolve())


## 2. Data

The [CodeXGLUE code-to-text](https://huggingface.co/datasets/google/code_x_glue_ct_code_to_text)
benchmark is derived from CodeSearchNet. Each Python example carries:

| field | content |
|---|---|
| `code` | the raw function source, **including** its docstring |
| `code_tokens` | the function tokenized with the docstring and comments **removed** |
| `docstring` | the full docstring |
| `docstring_tokens` | the tokenized first paragraph of the docstring (the summary) |

Following the benchmark's official recipe, the model input is `" ".join(code_tokens)` and the
target is `" ".join(docstring_tokens)`. Using `code` as the input would hand the model the answer
(see the leakage check in 2.3).

### 2.1 Load

In [ ]:
dataset = load_dataset("google/code_x_glue_ct_code_to_text", "python")
dataset


### 2.2 Subsample and build text fields

The full training split has ~250k functions, far more than a short GPU session needs for a
comparison study. Each split is **shuffled with a fixed seed before subsampling**: the dataset is
ordered by repository, so taking the first N rows would yield a handful of projects rather than a
representative sample. Examples with an empty code or docstring side are dropped.

In [ ]:
def take_subset(split, fraction, seed=SEED):
    n = max(1, int(len(split) * fraction))
    return split.shuffle(seed=seed).select(range(n))


def add_text_fields(example):
    return {
        "source": " ".join(example["code_tokens"]),
        "target": " ".join(example["docstring_tokens"]),
    }


def has_source_and_target(example):
    return len(example["code_tokens"]) > 0 and len(example["docstring_tokens"]) > 0


text = {}
for name in ("train", "validation", "test"):
    subset = take_subset(dataset[name], SUBSET_FRACTION).filter(has_source_and_target)
    text[name] = subset.map(add_text_fields)
    print(f"{name:>10}: {len(text[name]):>6} examples")

example = text["train"][0]
print("\nsource:", example["source"][:300], "...")
print("target:", example["target"])


### 2.3 Leakage check

A summarizer that can read the docstring inside its input is not summarizing, it is copying.
The check below measures how often the target summary occurs verbatim (ignoring case, whitespace
and punctuation) inside the model input, for the raw `code` field and for the `code_tokens`-based
source actually used here.

In [ ]:
def canonical(s):
    # compare on alphanumerics only, so tokenization differences ("DeepMind - style" vs "DeepMind-style")
    # cannot hide a verbatim copy
    return "".join(ch for ch in s.lower() if ch.isalnum())


def leak_rate(split, source_field):
    hits = 0
    for ex in split:
        hits += canonical(ex["target"]) in canonical(ex[source_field])
    return hits / len(split)


rows = []
for name in ("train", "validation", "test"):
    rows.append({"split": name,
                 "target inside raw `code`": f"{leak_rate(text[name], 'code'):.1%}",
                 "target inside `code_tokens` source": f"{leak_rate(text[name], 'source'):.1%}"})
display(pd.DataFrame(rows).set_index("split"))

assert leak_rate(text["test"], "source") < 0.05, "target text is leaking into the model input"


### 2.4 Length statistics

Used to confirm that `MAX_SOURCE_LEN` / `MAX_TARGET_LEN` truncate only a small tail of examples.

In [ ]:
lengths = pd.DataFrame({
    "source words": [len(s.split()) for s in text["train"]["source"]],
    "target words": [len(t.split()) for t in text["train"]["target"]],
})
display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
lengths["source words"].clip(upper=800).hist(bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("code length (words, clipped at 800)")
lengths["target words"].clip(upper=60).hist(bins=30, ax=axes[1], color="#DD8452")
axes[1].set_title("summary length (words, clipped at 60)")
plt.tight_layout()


## 3. Shared pipeline

Both models go through exactly the same code, so the comparison is fair.

- **Tokenization.** Targets are tokenized with `text_target=` so each tokenizer adds its own
  target-side special tokens (for PLBART this appends the `__en_XX__` language code, which is also
  what the decoder is started with at inference time).
- **Batching.** `DataCollatorForSeq2Seq` pads dynamically per batch and pads `labels` with `-100`,
  which the cross-entropy loss ignores.
- **Training.** AdamW with weight decay, linear warm-up/decay schedule, gradient clipping; after
  every epoch the validation set is decoded, and the checkpoint with the best validation BLEU is
  kept (early stopping once it stops improving).
- **Evaluation.** Beam-search generation; BLEU and ROUGE computed against the untruncated reference
  strings from the dataset.

### 3.1 Tokenization and data loaders

In [ ]:
def tokenize_split(split, tokenizer):
    def tokenize(batch):
        model_inputs = tokenizer(batch["source"], max_length=MAX_SOURCE_LEN, truncation=True)
        labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_LEN, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return split.map(tokenize, batched=True, remove_columns=split.column_names)


def build_loaders(tokenizer, batch_size):
    collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
    tokenized = {name: tokenize_split(split, tokenizer) for name, split in text.items()}
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True,
                            collate_fn=collator, generator=generator),
        "validation": DataLoader(tokenized["validation"], batch_size=batch_size, shuffle=False,
                                 collate_fn=collator),
        "test": DataLoader(tokenized["test"], batch_size=batch_size, shuffle=False,
                           collate_fn=collator),
    }


### 3.2 Generation and metrics

- **BLEU**: corpus-level BLEU-4 and the mean of smoothed sentence-level BLEU-4 (add-one smoothing
  on higher-order n-grams, the scheme of the CodeXGLUE evaluator, so it is comparable with the
  leaderboard). Both on a 0-100 scale.
- **ROUGE-1/2/L**: F-measure with stemming, 0-100 scale.

Every call creates its own scorer and aggregator, so metrics from different evaluations can never
mix.

In [ ]:
@torch.no_grad()
def generate_summaries(model, tokenizer, loader, **generate_kwargs):
    model.to(device).eval()
    outputs = []
    for batch in tqdm(loader, desc="generating", leave=False):
        generated = model.generate(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            **generate_kwargs,
        )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return [o.strip() for o in outputs]


def compute_bleu(predictions, references):
    hyps = [p.split() for p in predictions]
    refs = [[r.split()] for r in references]
    corpus = corpus_bleu(refs, hyps)
    # add-one smoothing on n>1 n-gram counts (Lin & Och, 2004), the scheme used by the CodeXGLUE evaluator
    smooth = SmoothingFunction().method2
    sentence = mean(sentence_bleu(r, h, smoothing_function=smooth) for r, h in zip(refs, hyps))
    return {"BLEU (corpus)": 100 * corpus, "BLEU (sentence avg)": 100 * sentence}


def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    aggregator = scoring.BootstrapAggregator()
    for pred, ref in zip(predictions, references):
        aggregator.add_scores(scorer.score(ref, pred))
    result = aggregator.aggregate()
    return {"ROUGE-1": 100 * result["rouge1"].mid.fmeasure,
            "ROUGE-2": 100 * result["rouge2"].mid.fmeasure,
            "ROUGE-L": 100 * result["rougeL"].mid.fmeasure}


def evaluate_generation(model, tokenizer, loader, references, label, n_examples=5, **generate_kwargs):
    predictions = generate_summaries(model, tokenizer, loader, **generate_kwargs)
    assert len(predictions) == len(references), (len(predictions), len(references))
    metrics = {**compute_bleu(predictions, references), **compute_rouge(predictions, references)}

    print(f"\n{label}")
    for name, value in metrics.items():
        print(f"  {name:<22}{value:6.2f}")
    print()
    for pred, ref in list(zip(predictions, references))[:n_examples]:
        print(f"  generated: {pred}")
        print(f"  reference: {ref}\n")
    return metrics, predictions


### 3.3 Training loop

Model selection follows the CodeXGLUE recipe: after every epoch the model decodes the validation
set and the checkpoint with the best validation BLEU is kept. Validation loss is still logged, but
it is a poor proxy for generation quality (it can rise while BLEU still improves), so it is not
used for selection or early stopping.

In [ ]:
def to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}


@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        total += model(**to_device(batch)).loss.item()
        n += 1
    return total / max(n, 1)


def train_model(model, tokenizer, loaders, run_name, val_references, generate_kwargs=None,
                num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, patience=PATIENCE):
    generate_kwargs = generate_kwargs or {}
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = len(loaders["train"]) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)

    best_bleu, best_state, bad_epochs, history = -math.inf, None, 0, []
    for epoch in range(1, num_epochs + 1):
        model.train()
        running = []
        progress = tqdm(loaders["train"], desc=f"{run_name} epoch {epoch}/{num_epochs}", leave=False)
        for batch in progress:
            loss = model(**to_device(batch)).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            running.append(loss.item())
            progress.set_postfix(loss=f"{mean(running[-50:]):.4f}")

        train_loss = mean(running)
        val_loss = evaluate_loss(model, loaders["validation"])
        val_predictions = generate_summaries(model, tokenizer, loaders["validation"], **generate_kwargs)
        val_bleu = compute_bleu(val_predictions, val_references)["BLEU (sentence avg)"]
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_bleu": val_bleu})
        improved = val_bleu > best_bleu
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val BLEU {val_bleu:5.2f}"
              + ("  <- best" if improved else ""))

        if improved:
            best_bleu, bad_epochs = val_bleu, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"no improvement for {patience} epochs, stopping early")
                break

    model.load_state_dict(best_state)
    save_dir = OUTPUT_DIR / run_name
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"restored best checkpoint (val BLEU {best_bleu:.2f}) and saved it to {save_dir}")
    return pd.DataFrame(history)


## 4. PLBART

[PLBART](https://arxiv.org/abs/2103.06333) is a BART-style denoising seq2seq model pre-trained on
Java, Python and natural language. `uclanlp/plbart-base` is the **pre-trained** checkpoint; it has
not seen the code-to-text task. (Hugging Face also hosts `uclanlp/plbart-python-en_XX`, which is
already fine-tuned on this very benchmark, so using it here would make the fine-tuning experiment
meaningless.)

PLBART works like mBART: the source sequence ends with the source language code (`__python__`),
the target with the target language code (`__en_XX__`), and the decoder is started with the target
language code at inference time.

In [ ]:
PLBART_CHECKPOINT = "uclanlp/plbart-base"

plbart_tokenizer = PLBartTokenizer.from_pretrained(PLBART_CHECKPOINT, src_lang="python", tgt_lang="en_XX")
plbart_model = PLBartForConditionalGeneration.from_pretrained(PLBART_CHECKPOINT)
plbart_loaders = build_loaders(plbart_tokenizer, batch_size=16)   # Kaggle: 16 fits comfortably on a 16 GB T4/P100

en_token_id = plbart_tokenizer.convert_tokens_to_ids("__en_XX__")
plbart_generate_kwargs = {"decoder_start_token_id": en_token_id}

# sanity check: the label sequence must end with the English language code
first_labels = plbart_loaders["validation"].dataset[0]["labels"]
assert first_labels[-1] == en_token_id, plbart_tokenizer.convert_ids_to_tokens(first_labels[-3:])
print("label tokens :", plbart_tokenizer.convert_ids_to_tokens(first_labels))
print("decoded label:", plbart_tokenizer.decode(first_labels, skip_special_tokens=True))
print(f"parameters   : {sum(p.numel() for p in plbart_model.parameters()) / 1e6:.0f}M")


In [ ]:
plbart_history = train_model(plbart_model, plbart_tokenizer, plbart_loaders, run_name="plbart",
                             val_references=text["validation"]["target"],
                             generate_kwargs=plbart_generate_kwargs)


In [ ]:
plbart_val_metrics, _ = evaluate_generation(
    plbart_model, plbart_tokenizer, plbart_loaders["validation"], text["validation"]["target"],
    label="PLBART | validation", **plbart_generate_kwargs)


The test set is scored exactly once, after training and model selection are finished.

In [ ]:
plbart_test_metrics, plbart_test_predictions = evaluate_generation(
    plbart_model, plbart_tokenizer, plbart_loaders["test"], text["test"]["target"],
    label="PLBART | test", **plbart_generate_kwargs)


In [ ]:
# release GPU memory before loading the next model (the best checkpoint is already on disk)
plbart_model.cpu()
del plbart_model
torch.cuda.empty_cache()


## 5. CodeT5

[CodeT5](https://arxiv.org/abs/2109.00859) is a T5 model pre-trained on code with
identifier-aware objectives. `Salesforce/codet5-base` is the pre-trained checkpoint (not the
summarization-fine-tuned `codet5-base-multi-sum`). It uses the same data and training code as
PLBART; T5 starts its decoder with the pad token by default, so no extra generation arguments are
needed. The batch size is halved because the model is ~1.6x larger.

In [ ]:
CODET5_CHECKPOINT = "Salesforce/codet5-base"

# The Hub repository stores its 100 sentinel tokens (<extra_id_0> ... <extra_id_99>) in a legacy
# dict format that transformers >= 5 fails to parse ("TypeError: Input must be a List[Union[str, AddedToken]]").
# Passing the list explicitly bypasses the legacy file and yields the identical vocabulary on every
# transformers version (32,100 tokens, <extra_id_99> = 32000 ... <extra_id_0> = 32099).
CODET5_SENTINEL_TOKENS = [f"<extra_id_{i}>" for i in range(99, -1, -1)]

codet5_tokenizer = AutoTokenizer.from_pretrained(CODET5_CHECKPOINT, additional_special_tokens=CODET5_SENTINEL_TOKENS)
codet5_model = AutoModelForSeq2SeqLM.from_pretrained(CODET5_CHECKPOINT)
assert len(codet5_tokenizer) == codet5_model.config.vocab_size == 32100
codet5_loaders = build_loaders(codet5_tokenizer, batch_size=8)   # Kaggle: 8 fits on a 16 GB T4/P100

first_labels = codet5_loaders["validation"].dataset[0]["labels"]
print("label tokens :", codet5_tokenizer.convert_ids_to_tokens(first_labels))
print("decoded label:", codet5_tokenizer.decode(first_labels, skip_special_tokens=True))
print(f"parameters   : {sum(p.numel() for p in codet5_model.parameters()) / 1e6:.0f}M")


In [ ]:
codet5_history = train_model(codet5_model, codet5_tokenizer, codet5_loaders, run_name="codet5",
                             val_references=text["validation"]["target"])


In [ ]:
codet5_val_metrics, _ = evaluate_generation(
    codet5_model, codet5_tokenizer, codet5_loaders["validation"], text["validation"]["target"],
    label="CodeT5 | validation")


In [ ]:
codet5_test_metrics, codet5_test_predictions = evaluate_generation(
    codet5_model, codet5_tokenizer, codet5_loaders["test"], text["test"]["target"],
    label="CodeT5 | test")


## 6. Results

### 6.1 Metrics

In [ ]:
results = pd.DataFrame({
    ("PLBART", "validation"): plbart_val_metrics,
    ("PLBART", "test"): plbart_test_metrics,
    ("CodeT5", "validation"): codet5_val_metrics,
    ("CodeT5", "test"): codet5_test_metrics,
}).T.round(2)
results.index.names = ["model", "split"]
results.to_csv(OUTPUT_DIR / "results.csv")
display(results)

ax = results.xs("test", level="split").T.plot.bar(figsize=(8, 3.5), rot=0, width=0.7,
                                                   color=["#4C72B0", "#DD8452"])
ax.set_title("test-set scores (higher is better)")
ax.set_ylabel("score (0-100)")
ax.legend(title=None, frameon=False)
plt.tight_layout()


### 6.2 Training curves

In [ ]:
fig, (ax_loss, ax_bleu) = plt.subplots(1, 2, figsize=(12, 3.6))
for name, hist, color in (("PLBART", plbart_history, "#4C72B0"), ("CodeT5", codet5_history, "#DD8452")):
    ax_loss.plot(hist["epoch"], hist["train_loss"], "--", color=color, alpha=0.6, label=f"{name} train")
    ax_loss.plot(hist["epoch"], hist["val_loss"], "-o", color=color, label=f"{name} validation")
    ax_bleu.plot(hist["epoch"], hist["val_bleu"], "-o", color=color, label=name)
    best = hist.loc[hist["val_bleu"].idxmax()]
    ax_bleu.annotate("kept", (best["epoch"], best["val_bleu"]), textcoords="offset points", xytext=(0, 8),
                     ha="center", color=color, fontsize=8)
ax_loss.set(xlabel="epoch", ylabel="cross-entropy loss", title="training and validation loss")
ax_bleu.set(xlabel="epoch", ylabel="validation BLEU (sentence avg)", title="validation BLEU per epoch")
ax_loss.legend(frameon=False)
ax_bleu.legend(frameon=False)
plt.tight_layout()


### 6.3 Side-by-side predictions

A few test examples where the two models disagree most, which is usually more informative than the
cases both get right.

In [ ]:
comparison = pd.DataFrame({
    "reference": text["test"]["target"],
    "PLBART": plbart_test_predictions,
    "CodeT5": codet5_test_predictions,
})
comparison.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

scorer = rouge_scorer.RougeScorer(["rougeL"])
comparison["PLBART rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for r, p in zip(comparison["reference"], comparison["PLBART"])]
comparison["CodeT5 rougeL"] = [scorer.score(r, p)["rougeL"].fmeasure for r, p in zip(comparison["reference"], comparison["CodeT5"])]
comparison["gap"] = (comparison["PLBART rougeL"] - comparison["CodeT5 rougeL"]).abs()

with pd.option_context("display.max_colwidth", 120):
    display(comparison.sort_values("gap", ascending=False).head(8)[["reference", "PLBART", "CodeT5"]])


## 7. Inference on new code

In practice the input is raw source code, so the same preprocessing that produced `code_tokens`
has to be applied: the docstring is removed with the `ast` module and the remaining code is
tokenized with Python's own tokenizer, dropping comments and layout tokens. The helper below wraps
this so the fine-tuned model can be used on any Python function.

In [ ]:
def strip_docstrings(source):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            body = node.body
            if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) \
                    and isinstance(body[0].value.value, str):
                node.body = body[1:] or [ast.Pass()]
    return ast.unparse(tree)


def code_to_tokens(source):
    skip = {py_tokenize.COMMENT, py_tokenize.NL, py_tokenize.NEWLINE, py_tokenize.INDENT,
            py_tokenize.DEDENT, py_tokenize.ENCODING, py_tokenize.ENDMARKER}
    tokens = py_tokenize.generate_tokens(io.StringIO(strip_docstrings(source)).readline)
    return [tok.string for tok in tokens if tok.type not in skip and tok.string.strip()]


@torch.no_grad()
def summarize(source, model, tokenizer, **generate_kwargs):
    model.to(device).eval()
    encoded = tokenizer(" ".join(code_to_tokens(source)), max_length=MAX_SOURCE_LEN,
                        truncation=True, return_tensors="pt").to(device)
    generated = model.generate(**encoded, max_new_tokens=MAX_TARGET_LEN, num_beams=NUM_BEAMS,
                               early_stopping=True, **generate_kwargs)
    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()


demo_function = '''
def check_file_balance(filename):
    """Checks if a file is balanced.

    A file is balanced if every open parenthesis has a matching closing parenthesis.
    """
    with open(filename, 'r') as f:
        stack = []
        for line in f:
            for char in line:  # scan character by character
                if char == '(':
                    stack.append(char)
                elif char == ')':
                    if not stack:
                        return False
                    stack.pop()
    return not stack
'''

print("model input :", " ".join(code_to_tokens(demo_function))[:200], "...")
print("CodeT5 says :", summarize(demo_function, codet5_model, codet5_tokenizer))


## 8. Methodology notes

Design decisions that keep the numbers honest. Each of these is easy to get wrong in a code
summarization setup, and several of them inflate scores silently rather than failing loudly.

| concern | decision in this notebook |
|---|---|
| Target visible in the input | Input is `code_tokens` (docstring and comments stripped), verified by the measured leakage check in 2.3. |
| Pre-trained vs. task-fine-tuned checkpoints | Both models start from pre-training only; task-fine-tuned Hub checkpoints for this benchmark are deliberately avoided. |
| Unrepresentative subset | Splits are shuffled with a fixed seed before subsampling instead of taking the first rows of a repository-sorted file. |
| Loss on padding | Labels are padded with `-100`, which the loss ignores. |
| Train/inference mismatch (PLBART) | Targets are tokenized in target mode (`text_target=`) so they end with `__en_XX__`, the same token the decoder is started with. |
| Overfitting | Model selection and early stopping on validation BLEU (the CodeXGLUE criterion); the best checkpoint is restored, not the last epoch. |
| Test-set reuse | Model selection uses only the validation set; the test set is decoded and scored once per model. |
| Metric bookkeeping | Fresh scorer/aggregator per evaluation; references are the untruncated dataset strings, not decoded label tensors. |
| Reproducibility | Global seed, seeded shuffling and data-loader order, all hyper-parameters in one configuration cell. |


## 9. Conclusion and next steps

Both PLBART and CodeT5 learn to produce fluent one-line summaries of Python functions from a small
fraction of CodeXGLUE within a single Kaggle GPU session. With the leakage removed, the scores in
Section 6 reflect genuine summarization quality and can be compared with the published CodeXGLUE
code-to-text leaderboard, where the full-data fine-tuned models reach smoothed BLEU of roughly
19-20 on Python.

Natural extensions:

- **More data.** Increase `SUBSET_FRACTION`; scores on this benchmark keep improving well past 10%.
- **Other languages.** The dataset has Java, JavaScript, Go, PHP and Ruby configurations; PLBART's
  `src_lang` must be set accordingly.
- **Decoding.** Tune `NUM_BEAMS`, add length penalties or n-gram blocking.
- **Human-oriented evaluation.** BLEU and ROUGE reward lexical overlap; a small manual rating of
  fluency and correctness would complement them.

### References
- Lu et al., *CodeXGLUE: A Machine Learning Benchmark Dataset for Code Understanding and Generation*, 2021.
- Ahmad et al., *Unified Pre-training for Program Understanding and Generation* (PLBART), NAACL 2021.
- Wang et al., *CodeT5: Identifier-aware Unified Pre-trained Encoder-Decoder Models for Code Understanding and Generation*, EMNLP 2021.
- Husain et al., *CodeSearchNet Challenge: Evaluating the State of Semantic Code Search*, 2019.
